# Õppetund 13 - Agentide mälu Cognee teadmistegraafikutega


## Ülesseadmine

See märkmik demonstreerib, kuidas ehitada intelligentne **kodeerimisabi** püsiva mäluga, kasutades [**Cognee**](https://www.cognee.ai/) teadmiste graafe ja **Microsoft Agent Frameworki** (MAF).

Cognee teisendab struktureerimata teksti struktureeritud, päringuvõimeliseks teadmiste graafiks, mida toetavad vektorsisseviited — andes teie agendile rikkaliku, suhetega teadlikku pikaajalist mälu.

### Mida Sa Õpid
1. **Ehitada teadmiste graafe**: Teisendada arendajate profiile ja parimaid tavasid struktureeritud, päringuvõimeliseks teadmiseks.
2. **Integreerida Cognee MAF-iga**: Kasutada `@tool` funktsioone, et lasta MAF agentidel pärida Cognee teadmiste graafi.
3. **Sessiooniteadlikud vestlused**: Hoida konteksti üle mitme küsimuse sama sessiooni jooksul.
4. **Pikaajaline mälu**: Säilitada olulist teadmist sessioonide vahel ja tuua see välja uutes vestlustes.

### Eeldused
- Python 3.9+
- Kohalikus keskkonnas töötav Redis (`docker run -d -p 6379:6379 redis`) sessioonihalduseks
- LLM API võti (nt OpenAI) — seadista `LLM_API_KEY` failis `.env`
- `CACHING=true` failis `.env` (nõutud Cognee sessioonide jaoks)
- Azure AI Foundry projekt koos kasutusele võetud vestlusmudeliga
- `AZURE_AI_PROJECT_ENDPOINT` ja `AZURE_AI_MODEL_DEPLOYMENT_NAME` failis `.env`
- Azure CLI autoriseeritud (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Agentmälutüübid

See märkmik uurib samu kolme mälutüüpi nagu põhiosas 13. õppetüki märkmikus, kuid kasutab pikaajalise mäluna Cogneet:

| Mälutüüp | Mehhanism | Eluiga |
|---|---|---|
| **Töömälu** | `agent.create_session()` (MAF) | Üks vestlus |
| **Lühiajaline** | Cognee seansi vahemälu (Redis)  | Üks seanss |
| **Pikaajaline** | Cognee teadmiste graafik + vektorid | Püsiv |

### Cognee mälu arhitektuur
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Valmista ette Cognee salvestusruum


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Osa 1 — Teadmusbaasi loomine

Me kogume kolme tüüpi andmeid, et luua oma kodeerimisassistendile ulatuslik teadmiste baas:

1. **Arendaja profiil** — isiklikud teadmised ja tehniline taust
2. **Python parimad tavad** — Pythoni Zen praktiliste juhistega
3. **Ajaloolised vestlused** — varasemad arendajate ja tehisintellekti assistentide küsimused ja vastused


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Teadmiste graafi visualiseerimine

Cognee suudab kuvada interaktiivse HTML-visualiseeringu tuvastatud üksustest ja suhetest.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Rikasta mälu Memify abil

`memify()` analüüsib teadmiste graafi ja genereerib intelligentseid reegleid — tuvastades mustreid, parimaid tavasid ja seoseid kontseptsioonide vahel.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Osa 2 — MAF-agent Cognee tööriistadega

Nüüd loome MAF-agendi, kes suudab pärida Cognee teadmiste graafikut `@tool` funktsioonide kaudu. See võimaldab agendil kasutada täisväärtuslikku graafirakenduslikku semantilist otsingut, säilitades samal ajal vestluse konteksti sessioonide kaudu.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Töömäluga seansside puhul

`AgentSession` (loodud `agent.create_session()` abil) pakub töömäluruumi seansi jooksul. Agent saab viidata varasematele sõnumitele ning samal ajal pärida Cognee pikaajalist teadmiste graafi.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Uus seanss — Pikaajaline mälu säilib

Uue seansi alustamine tühjendab töömälus oleva teabe, kuid Cognee teadmiste graaf on endiselt saadaval. Agent saab täiesti uues vestluses taaskasutada sama pikaajalist teadmist.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Kokkuvõte

Selles märkmikus ehitasite kodeerimisabilise, mis kombineerib **MAF-i töömälu** (`agent.create_session()`) ja **Cognee pikaajalise teadmiste graafi**.

### Mida õppisite
1. **Teadmiste graafi ehitamine**: Cognee töötleb struktureerimata teksti ja ehitab graafi + vektormälu.
2. **Graafi rikastamine memify abil**: Tuletatud faktid ja rikkalikumad seosed teie olemasoleva graafi peal.
3. **MAF + Cognee integratsioon**: `@tool` funktsioonid võimaldavad MAF agendil loomulikult pärida Cognee graafi.
4. **Töömälu + pikaajaline mälu**: `AgentSession` (kaudu `agent.create_session()`) pakub sessiooni konteksti, samas kui Cognee annab püsiva teadmise.
5. **Filtreeritud otsing NodeSets abil**: Suunatud teadmiste graafi spetsiifilistele alamhulkadele (nt ainult põhimõtted).

### Peamised teadmised
- **Cognee** muudab toorteksti struktureeritud, suhetest teadlikuks mäluks — võimsam kui lame vektoripood.
- **`@tool` funktsioonid** sillutavad MAF agentide ja väliste teadmussüsteemide puhta ühenduse.
- **`AgentSession`** (kaudu `agent.create_session()`) hoiab vestluse konteksti eraldi pikaajalisest teadmisest.
- Ühtne teadmiste graaf teenindab mitut sessiooni ja agenti.

### Reaalsed rakendused
- **Arendaja abikoerad**: koodi ülevaatus, intsidentide analüüs, arhitektuuri assistendid
- **Kliendisuhtluse assistendid**: tugiteenused tooteteadmete, KKK ja kliendihaldusmärkmete põhjal
- **Sisemised eksperdi asistendid**: poliitika, õigusalased või turvaseiret toetavad abistajad juhiste baasil
- **Ühtne andmekiht**: struktureeritud ja struktureerimata andmete kombineerimine üheks päringugraafiks

### Järgmised sammud
- Katseta ajateadlikkust Cognees
- Määra domeenipõhine OWL ontoloogia graafi kvaliteedile
- Lisa kasutajate tagasiside silmused parendamaks otsingut aja jooksul
- Skaala mitme agendi süsteemide jaoks, mis jagavad sama Cognee mälukihti


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastutusest loobumine**:
See dokument on tõlgitud AI tõlketeenuse [Co-op Translator](https://github.com/Azure/co-op-translator) abil. Kuigi püüame täpsust, tuleb arvestada, et automaatsed tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Kriitilise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlke kasutamisest tingitud võimalikest arusaamatustest või tõlgendustest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
